<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Manh2307/PDF-Rag-Assistant/blob/demo-Ti%E1%BA%BFn/Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cài đặt các thư viện hệ thống cần thiết cho đồ họa
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw

# Tải và cài đặt công cụ Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-d

In [2]:
import os, subprocess, time

# Tắt Flash Attention để tránh lỗi sập mô hình trên GPU T4 của Colab
os.environ['OLLAMA_FLASH_ATTENTION'] = 'false'

# Khởi chạy Ollama server ở chế độ chạy ngầm
subprocess.Popen(["ollama", "serve"])
time.sleep(30)

# Tiến hành tải 2 mô hình theo đúng tài liệu dự án
print("Đang tải mô hình bge-m3 (Vui lòng đợi)...")
# Tải mô hình Embedding (chuyển text thành vector)
!ollama pull bge-m3

print("Đang tải mô hình vicuna:7b (Mô hình này khá nặng, bạn đợi vài phút nhé)...")
# Tải mô hình Embedding (chuyển text thành vector)
!ollama pull vicuna:7b-v1.5-q5_1

print("Đã tải xong các mô hình AI.")

Đang tải mô hình bge-m3 (Vui lòng đợi)...

Đang tải mô hình vicuna:7b (Mô hình này khá nặng, bạn đợi vài phút nhé)...

Đã tải xong các mô hình AI.


In [3]:
!pip install pypdf chromadb ollama

import pypdf
import chromadb
import ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    

In [4]:
#1. Đọc PDF -> tạo chuỗi văn bản

def load_pdf(file_path: str) -> str:
    """
    Đọc file PDF từ đường dẫn cho trước và trích xuất toàn bộ văn bản.

    Args:
        file_path (str): Đường dẫn tới file PDF cần đọc.

    Returns:
        str: Chuỗi văn bản tổng hợp từ tất cả các trang của PDF.
    """
    # Khởi tạo đối tượng PdfReader từ thư viện pypdf
    reader = pypdf.PdfReader(file_path)

    pages_text = []

    # Duyệt qua từng trang của tài liệu
    for page in reader.pages:
        # Trích xuất text từ trang hiện tại
        text = page.extract_text()

        # Phòng thủ lỗi: Nếu trang chỉ có ảnh hoặc lỗi không lấy được text,
        # extract_text() sẽ trả về None. Ta thay thế bằng chuỗi rỗng "".
        if text is None:
            text = ""

        pages_text.append(text)

    # Gộp toàn bộ văn bản của các trang lại với nhau, phân tách bằng dấu xuống dòng
    full_text = "\n".join(pages_text)
    return full_text

#2. Chunking
def chunk_text(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> list:
    """
    Cắt một chuỗi văn bản lớn thành các đoạn nhỏ hơn (chunks) dựa trên
    kích thước chunk_size và độ trùng lặp chunk_overlap.

    Args:
        text (str): Chuỗi văn bản đầu vào khổng lồ.
        chunk_size (int): Số lượng ký tự tối đa của một chunk.
        chunk_overlap (int): Số lượng ký tự trùng lặp gối đầu giữa các chunk liên tiếp.

    Returns:
        list: Danh sách các chuỗi văn bản đã được cắt nhỏ.
    """
    chunks = []
    words = text.split()  # Tách văn bản thành danh sách các từ dựa trên khoảng trắng
    full_text_cleaned = " ".join(words)  # Chuẩn hóa văn bản loại bỏ khoảng trắng thừa

    start_idx = 0
    text_length = len(full_text_cleaned)

    # Vòng lặp dịch chuyển cửa sổ cắt (Sliding Window)
    while start_idx < text_length:
        # Xác định vị trí kết thúc dự kiến của chunk hiện tại
        end_idx = start_idx + chunk_size

        # Lấy ra đoạn văn bản trong khung cửa sổ
        chunk = full_text_cleaned[start_idx:end_idx]
        chunks.append(chunk)

        # Dịch chuyển vị trí bắt đầu cho chunk tiếp theo.
        # Khoảng dịch chuyển bằng (Kích thước chunk - Độ trùng lặp)
        start_idx += (chunk_size - chunk_overlap)

    return chunks



In [6]:
file_path = "main.pdf"
full_text = load_pdf(file_path)
print(f'Tổng số ký tự: {len(full_text)}')
print(full_text[4000:5000])
chunks = chunk_text(full_text)
print(f'Tổng số chunk: {len(chunks)}')
print(f'Chuỗi văn bản mẫu: {chunks[25]}')

Tổng số ký tự: 36646
 chỉ đơn giản là giải phương trình đạo hàm bằng không.
Một điểm làm cho gradient triệt tiêu có thể là điểm cực tiểu, cực đại hoặc điểm
yên ngựa. Vì thế, các điều kiện tối ưu đóng vai trò quan trọng trong việc nhận
diện và phân loại các điểm dừng của hàm số. Điều kiện cần bậc nhất giúp xác
định các điểm ứng viên thông qua phương trình∇f(x ∗) = 0.
Tuy nhiên, để kết luận chính xác hơn về bản chất của điểm dừng, ta cần xét
thêm các điều kiện bậc hai thông qua ma trận Hess∇ 2f(x ∗).
Trong báo cáo này, nhóm trình bày một cách hệ thống các điều kiện tối ưu cho
bài toán quy hoạch phi tuyến không ràng buộc. Nội dung bao gồm điều kiện cần
bậc nhất, điều kiện cần bậc hai, điều kiện đủ bậc hai, khái niệm ma trận xác định
dương, ma trận nửa xác định dương và tiêu chuẩn Sylvester. Bên cạnh đó, báo cáo
cũng xét trường hợp đặc biệt của hàm lồi, trong đó mọi điểm dừng của hàm lồi
khả vi đều là nghiệm tối ưu toàn cục.
Sau phần cơ sở lý thuyết, nhóm áp dụng các kết quả trên vào bài t

In [7]:
#Vector hóa và lưu trữ vào Vector database

def create_vector_db(chunks: list, collection_name: str = "rag_collection") -> chromadb.Collection:
    """
    Khởi tạo ChromaDB, tạo embedding cho các đoạn chunks thông qua Ollama
    và lưu chúng vào một Collection.

    Args:
        chunks (list): Danh sách các đoạn văn bản đã được cắt nhỏ từ Phần II.
        collection_name (str): Tên của nhóm lưu trữ dữ liệu trong ChromaDB.

    Returns:
        chromadb.Collection: Đối tượng Collection đã chứa dữ liệu và sẵn sàng truy vấn.
    """
    # 1. Khởi tạo Client của ChromaDB (Mặc định chạy trên RAM - In Memory)
    # Hướng dẫn mở rộng: Có thể dùng PersistentClient để lưu trực tiếp xuống ổ cứng
    chroma_client = chromadb.Client()

    # 2. Định nghĩa hàm tự tạo Embedding bằng mô hình bge-m3 thông qua Ollama.
    # Thư viện chromadb yêu cầu hàm embedding phải nhận vào một danh sách các chuỗi (texts)
    # và trả về một danh sách các vector tương ứng.
    class OllamaEmbeddingFunction:

      def _embed(self, texts):
        embeddings = []

        for text in texts:
            response = ollama.embeddings(
                model="bge-m3",
                prompt=text
            )
            embeddings.append(response["embedding"])

        return embeddings

      def __call__(self, input):
        return self._embed(input)

      def embed_documents(self, input):
        return self._embed(input)

      def embed_query(self, input):
        if isinstance(input, str):
            input = [input]

        return self._embed(input)

      def name(self):
        return "ollama-bge-m3"

    embedding_function = OllamaEmbeddingFunction()

    # 3. Tạo hoặc lấy lại một Collection (tương tự như một Bảng trong SQL) kèm theo hàm Embedding trên
    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_function
    )

    # 4. Chuẩn bị dữ liệu để nạp vào Database:
    # Cần cung cấp ID độc nhất cho mỗi chunk và lưu chính văn bản gốc đó vào trường 'documents'
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    # Nạp dữ liệu vào Collection.
    # Khi hàm .add() được gọi, ChromaDB sẽ tự động kích hoạt hàm embedding_function
    # để sinh ra vector cho toàn bộ `documents` trước khi lưu trữ.
    collection.add(
        documents=chunks,
        ids=ids
    )

    return collection

In [8]:
collection = create_vector_db(chunks)
print(f"Tổng số lượng chunk: {collection.count()}")
print(f"Vector dimension: {collection.get(include=['embeddings'])['embeddings'][0].shape}")
print(f"Vector example: {collection.get(include=['embeddings'])['embeddings'][0][:5]}")


Tổng số lượng chunk: 46
Vector dimension: (1024,)
Vector example: [-0.96845698 -0.16900927 -0.85293299 -0.2868315  -1.24764049]


In [9]:
# Truy vấn

def retrieve_context(query: str, collection: chromadb.Collection, n_results: int = 4) -> list:
    """
    Nhận câu hỏi từ người dùng, tự động vector hóa câu hỏi và truy vấn
    ra các đoạn văn bản liên quan nhất từ ChromaDB.

    Args:
        query (str): Câu hỏi thô từ người dùng nhập vào.
        collection (chromadb.Collection): Đối tượng kho lưu trữ dữ liệu đã tạo từ Phần III.
        n_results (int): Số lượng đoạn văn bản liên quan nhất cần lấy ra.

    Returns:
        list: Danh sách các chuỗi văn bản (chunks) có ngữ nghĩa sát với câu hỏi nhất.
    """
    # Thực hiện truy vấn trong Collection dữ liệu
    # ChromaDB sẽ tự dùng hàm OllamaEmbeddingFunction (đã khai báo ở Phần III)
    # để tự động embedding câu hỏi `query_texts` trước khi tìm kiếm.
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    # Kết quả trả về của ChromaDB có cấu trúc dạng dictionary phức tạp:
    # results['documents'] là một list của các list: [[chunk1, chunk2, chunk3, chunk4]]
    # Ta lấy phần tử đầu tiên [0] để thu được danh sách phẳng chứa các đoạn text văn bản gốc.
    retrieved_chunks = results['documents'][0]

    return retrieved_chunks



In [11]:
question = "Thế nào là Ma trận xác định dương?"
retrieved_chunks = retrieve_context(question, collection)
print(retrieved_chunks)


['finite) ChoAlà ma trận thực vuông đối xứng cấpn×n. Ta nóiAlànửa xác định dương(ký hiệuA⪰0) nếu: x⊤Ax≥0∀x∈R n. Định nghĩa 2.4.2 — Ma trận xác định dương (Positive Definite) Ma trận đối xứngAđược gọi làxác định dương(ký hiệuA≻0) nếu: x⊤Ax>0∀x∈R n,x̸=0. Định nghĩa 2.4.3 — Ma trận nửa xác định âm và xác định âm Tương tự, ta định nghĩa: •Alànửa xác định âm(A⪯0) nếux ⊤Ax≤0với mọix∈R n. •Alàxác định âm(A≺0) nếux ⊤Ax<0với mọix̸=0. •Alàbất định(indefinite) nếu tồn tạix,ysao chox ⊤Ax>0vày ⊤Ay<0. 1.2 Tiêu Chuẩn Sylvester Định nghĩa 2.4.4 — Định thức con chính (Leading Principal Minor) Cho ma trận đối xứngA∈R n×n.Định thức con chính thứk(ký hiệu∆ k) là định thức của ma trận conk×kở góc trên trái củaA: ∆k = det(Ak) = det \x00 [aij]1≤i,j≤k \x01 , k= 1,2, . . . , n. Định lý 2.4.2 — Tiêu chuẩn Sylvester (cho xác định dương) ChoAlà ma trận đối xứng vuông thực cấpn×n.Alà xác định dương khi và chỉ khi tất cả các định thức con chính đều dương [?]: A≻0⇐ ⇒∆ 1 >0,∆ 2 >0, . . . ,∆n >0. Cơ sở tối ưu (AS2065)

In [10]:
#Connect LLM sinh câu trả ời

def generate_answer(query: str, retrieved_chunks: list, model_name: str = "vicuna:7b-v1.5-q5_1") -> str:
    """
    Kết hợp câu hỏi và ngữ cảnh tìm được thành một Prompt nghiêm ngặt,
    sau đó gọi mô hình LLM để sinh câu trả lời chính xác.

    Args:
        query (str): Câu hỏi của người dùng.
        retrieved_chunks (list): Danh sách các đoạn văn bản liên quan thu được từ Phần IV.
        model_name (str): Tên mô hình LLM đang chạy trên Ollama.

    Returns:
        str: Câu trả lời cuối cùng từ chatbot.
    """
    # 1. Gộp các đoạn chunks lại thành một khối văn bản duy nhất để làm ngữ cảnh
    context = "\n---\n".join(retrieved_chunks)

    # 2. Xây dựng cấu trúc Prompt nghiêm ngặt (Prompt Engineering)
    # Câu lệnh phân định rõ ràng đâu là thông tin nền (Context) và đâu là câu hỏi (Question)
    prompt = f"""Bạn là một trợ lý hỏi đáp tài liệu học tập thông minh và trung thực.
Hãy sử dụng các đoạn ngữ cảnh (Context) được cung cấp dưới đây để trả lời câu hỏi (Question).

---
NGỮ CẢNH (CONTEXT):
{context}
---

CÂU HỎI (QUESTION):
{query}

---
YÊU CẦU:
- Trả lời ngắn gọn, rõ ràng, đi thẳng vào vấn đề.
- Chỉ dựa vào thông tin có trong phần NGỮ CẢNH phía trên.
- Nếu thông tin trong NGỮ CẢNH không có hoặc không đủ để trả lời, hãy nói thẳng là "Tôi không biết câu trả lời dựa trên tài liệu đã cung cấp", TUYỆT ĐỐI KHÔNG ĐƯỢC BỊA ĐẶT thông tin.

TRẢ LỜI:"""

    # 3. Gọi mô hình LLM thông qua API của Ollama
    # Đặt temperature = 0 để loại bỏ tính ngẫu nhiên, giúp câu trả lời chính xác và thực tế nhất
    response = ollama.generate(
        model=model_name,
        prompt=prompt,
        options={
            "temperature": 0.0
        }
    )

    # Kết quả trả về chứa nội dung văn bản ở trường 'response'
    answer = response['response']

    return answer

In [12]:
answer = generate_answer(question, retrieved_chunks)
print(f'Câu hỏi: {question}')
print(f'Trả lời: {answer}')

Câu hỏi: Thế nào là Ma trận xác định dương?
Trả lời: Ma trận xác định dương là một ma trận đối xứng vuông thực cấp n×n, trong đó tất cả các định thức con chính (principal minors) đều dương.
